In [1]:
# colab_04_xgboost_kiyas.py
# Global XGBoost vs Global NN+embedding - AYNI pencere, AYNI metrikler
# Cikti: model/xgb_kiyas.json + model/xgb_kiyas.png

import os, json, time
import numpy as np
import pandas as pd
import xgboost as xgb
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

DIZIN = "/content/drive/MyDrive/Colab Notebooks/datasets/rossman"
HAZIRLIK = f"{DIZIN}/hazirlik"
CIKTI = f"{DIZIN}/model"

KANTIL_DE = True          # p10/p90 da egitilsin mi (kapsama kiyasi icin)
ROUND, ERKEN = 3000, 100
print("xgboost:", xgb.__version__)


xgboost: 3.4.1


In [3]:
# ------------------------------------------------------------------
# 1. VERI
# ------------------------------------------------------------------
meta = json.load(open(f"{HAZIRLIK}/meta.json"))
UFUK = meta["ufuk"]

def oku(ad):
    return pd.read_parquet(f"{HAZIRLIK}/xgb_{ad}.parquet")

egitim, val, test = oku("egitim"), oku("val"), oku("test")
print(f"egitim {egitim.shape} | val {val.shape} | test {test.shape}")
print(f"tarih araliklari: egitim {egitim.tarih.max().date()} | "
      f"val {val.tarih.min().date()}..{val.tarih.max().date()} | "
      f"test {test.tarih.min().date()}..{test.tarih.max().date()}")
assert egitim.tarih.max() < val.tarih.min(), "SIZINTI: egitim val'e tasiyor!"
assert val.tarih.max() < test.tarih.min(), "SIZINTI: val test'e tasiyor!"

OZELLIK = [c for c in egitim.columns if c not in ("tarih", "hedef")]
print(f"{len(OZELLIK)} ozellik: {OZELLIK}")

def XY(df):
    return df[OZELLIK], np.log1p(df["hedef"].values)   # NN gibi log uzayda ogren

Xtr, ytr = XY(egitim); Xva, yva = XY(val); Xte, yte = XY(test)
dtr = xgb.DMatrix(Xtr, ytr); dva = xgb.DMatrix(Xva, yva); dte = xgb.DMatrix(Xte, yte)
del Xtr, Xva, Xte

egitim (2191495, 25) | val (130570, 25) | test (26845, 25)
tarih araliklari: egitim 2015-05-17 | val 2015-05-25..2015-07-03 | test 2015-07-04..2015-07-31
23 ozellik: ['magaza', 'h', 'dow', 'ay', 'gun', 'hafta', 'promo', 'okul', 'tatil', 'promo2', 'magaza_tipi', 'urun_yelpazesi', 'log_rakip_mesafe', 'rakip_ay', 'promo2_hafta', 'ort7', 'ort28', 'ort56', 'std28', 'acik_oran28', 'dow_ort', 'gecen_yil', 'gecen_yil_acik']


In [4]:
# ------------------------------------------------------------------
# 2. METRIKLER (colab_02 ile BIREBIR AYNI tanimlar, ham EUR)
# ------------------------------------------------------------------
def smape(g, t):
    payda = (np.abs(g) + np.abs(t)) / 2.0
    ok = payda > 1e-6
    return 100.0 * np.mean(np.abs(g[ok] - t[ok]) / payda[ok])

def wmape(g, t):   return 100.0 * np.abs(g - t).sum() / max(np.abs(g).sum(), 1e-6)
def yanlilik(g, t): return 100.0 * (t.sum() - g.sum()) / max(g.sum(), 1e-6)
def kapsama(g, a, u): return 100.0 * np.mean((g >= a) & (g <= u))

In [5]:
# ------------------------------------------------------------------
# 3. EGITIM - p50
# ------------------------------------------------------------------
par = dict(objective="reg:squarederror", eval_metric="rmse",
           max_depth=8, eta=0.05, subsample=0.8, colsample_bytree=0.8,
           min_child_weight=5, tree_method="hist", device="cuda", seed=42)

t0 = time.time()
model = xgb.train(par, dtr, ROUND, evals=[(dva, "val")],
                  early_stopping_rounds=ERKEN, verbose_eval=200)
sure = (time.time() - t0) / 60
print(f"\np50 egitildi: {model.best_iteration+1} round, {sure:.1f} dk")

def tahmin(m, d):
    return np.expm1(m.predict(d, iteration_range=(0, m.best_iteration + 1)))

p50_va, p50_te = tahmin(model, dva), tahmin(model, dte)
g_va, g_te = val["hedef"].values, test["hedef"].values

[0]	val-rmse:0.39986
[200]	val-rmse:0.11463
[285]	val-rmse:0.11474

p50 egitildi: 186 round, 0.1 dk


In [6]:
# ------------------------------------------------------------------
# 4. EGITIM - p10 / p90 (opsiyonel, kapsama kiyasi icin)
# ------------------------------------------------------------------
kantil = {}
if KANTIL_DE:
    try:
        for q in [0.10, 0.90]:
            pq = dict(par); pq.update(objective="reg:quantileerror", quantile_alpha=q)
            pq.pop("eval_metric", None)
            mq = xgb.train(pq, dtr, ROUND, evals=[(dva, "val")],
                           early_stopping_rounds=ERKEN, verbose_eval=False)
            kantil[q] = (tahmin(mq, dva), tahmin(mq, dte))
            print(f"q{int(q*100)} egitildi: {mq.best_iteration+1} round")
    except Exception as e:
        print(f"kantil egitimi atlandi ({type(e).__name__}: {e}) - xgboost 2.0+ gerekir")
        kantil = {}

q10 egitildi: 588 round
q90 egitildi: 1175 round


In [7]:
# ------------------------------------------------------------------
# 5. DEGERLENDIRME
# ------------------------------------------------------------------
def rapor(ad, g, t, df, alt=None, ust=None):
    uf = np.array([smape(g[df.h.values == h], t[df.h.values == h]) for h in range(1, UFUK+1)])
    c = dict(smape=float(smape(g, t)), wmape=float(wmape(g, t)),
             yanlilik=float(yanlilik(g, t)),
             ufuk_smape=[float(x) for x in uf],
             h1=float(uf[0]), h28=float(uf[-1]))
    if alt is not None:
        c["kapsama"] = float(kapsama(g, alt, ust))
    mg = df.assign(g=g, t=t).groupby("magaza").apply(
        lambda x: smape(x.g.values, x.t.values), include_groups=False)
    c["magaza_smape_p50"] = float(np.percentile(mg, 50))
    c["magaza_smape_p95"] = float(np.percentile(mg, 95))
    c["magaza_smape"] = {int(k): float(v) for k, v in mg.items()}
    print(f"\n=== XGBoost {ad} ===")
    print(f"sMAPE {c['smape']:.2f}% | WMAPE {c['wmape']:.2f}% | yanlilik {c['yanlilik']:+.2f}%"
          + (f" | kapsama {c['kapsama']:.1f}%" if alt is not None else ""))
    print(f"1. gun {c['h1']:.2f}% | 28. gun {c['h28']:.2f}% | "
          f"magaza p50 {c['magaza_smape_p50']:.2f}% p95 {c['magaza_smape_p95']:.2f}%")
    return c

x_val = rapor("VAL", g_va, p50_va, val,
              *( (kantil[0.10][0], kantil[0.90][0]) if kantil else (None, None) ))
x_test = rapor("TEST", g_te, p50_te, test,
               *( (kantil[0.10][1], kantil[0.90][1]) if kantil else (None, None) ))



=== XGBoost VAL ===
sMAPE 8.75% | WMAPE 8.65% | yanlilik +0.35% | kapsama 75.1%
1. gun 8.99% | 28. gun 8.46% | magaza p50 8.44% p95 12.06%

=== XGBoost TEST ===
sMAPE 9.07% | WMAPE 8.88% | yanlilik +0.58% | kapsama 75.9%
1. gun 17.51% | 28. gun 9.47% | magaza p50 8.60% p95 13.36%


In [8]:
# ------------------------------------------------------------------
# 6. NN ile YAN YANA
# ------------------------------------------------------------------
nn = json.load(open(f"{CIKTI}/global_model_sonuc.json"))
kal = json.load(open(f"{CIKTI}/kalibrasyon.json"))

print("\n" + "="*66)
print("GLOBAL NN + EMBEDDING  vs  GLOBAL XGBOOST   (TEST 07-04..07-31)")
print("="*66)
print(f"{'metrik':<26}{'NN':>13}{'XGBoost':>13}{'fark':>12}")
for ad, a, b in [("sMAPE (p50)", nn["test"]["smape"], x_test["smape"]),
                 ("WMAPE", nn["test"]["wmape"], x_test["wmape"]),
                 ("1. gun sMAPE", nn["test"]["ufuk_smape"][0], x_test["h1"]),
                 ("28. gun sMAPE", nn["test"]["ufuk_smape"][-1], x_test["h28"]),
                 ("magaza sMAPE p95", np.percentile(list(nn["test"]["magaza_smape"].values()), 95),
                  x_test["magaza_smape_p95"])]:
    print(f"{ad:<26}{a:>12.2f}%{b:>12.2f}%{b-a:>+11.2f}p")
print(f"{'yanlilik':<26}{nn['test']['yanlilik']:>+12.2f}%{x_test['yanlilik']:>+12.2f}%")
if "kapsama" in x_test:
    print(f"{'p10-p90 kapsama':<26}{kal['test']['kapsama_sonra']:>12.1f}%{x_test['kapsama']:>12.1f}%"
          f"   (NN kalibre)")
print(f"\nNot: XGBoost 88 origin'den, NN 264 origin'den beslendi (XGB_ORIGIN_ATLA=3). "
      f"Takvim araligi ayni.")

# ortak/ayrisan magazalar
nn_mg = {int(k): v for k, v in nn["test"]["magaza_smape"].items()}
ortak = sorted(set(nn_mg) & set(x_test["magaza_smape"]))
fark = np.array([x_test["magaza_smape"][m] - nn_mg[m] for m in ortak])
print(f"\nmagaza bazli: XGBoost {int((fark<0).sum())} magazada daha iyi, "
      f"NN {int((fark>0).sum())} magazada daha iyi (toplam {len(ortak)})")
korelasyon = float(np.corrcoef([nn_mg[m] for m in ortak],
                               [x_test['magaza_smape'][m] for m in ortak])[0,1])
print(f"iki modelin magaza bazli hata korelasyonu: {korelasyon:.3f}  "
      f"<- yuksekse zorluk MAGAZADAN geliyor, modelden degil")


GLOBAL NN + EMBEDDING  vs  GLOBAL XGBOOST   (TEST 07-04..07-31)
metrik                               NN      XGBoost        fark
sMAPE (p50)                       8.34%        9.07%      +0.72p
WMAPE                             8.12%        8.88%      +0.76p
1. gun sMAPE                     13.96%       17.51%      +3.55p
28. gun sMAPE                     9.45%        9.47%      +0.01p
magaza sMAPE p95                 13.08%       13.36%      +0.28p
yanlilik                         -0.76%       +0.58%
p10-p90 kapsama                   77.4%        75.9%   (NN kalibre)

Not: XGBoost 88 origin'den, NN 264 origin'den beslendi (XGB_ORIGIN_ATLA=3). Takvim araligi ayni.

magaza bazli: XGBoost 315 magazada daha iyi, NN 800 magazada daha iyi (toplam 1115)
iki modelin magaza bazli hata korelasyonu: 0.872  <- yuksekse zorluk MAGAZADAN geliyor, modelden degil


In [9]:
# ------------------------------------------------------------------
# 7. GRAFIK
# ------------------------------------------------------------------
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
h = np.arange(1, UFUK+1)
ax[0].plot(h, nn["test"]["ufuk_smape"], "o-", ms=3, label="NN + embedding")
ax[0].plot(h, x_test["ufuk_smape"], "s-", ms=3, label="XGBoost")
ax[0].set_xlabel("ufuk (gun)"); ax[0].set_ylabel("sMAPE %")
ax[0].set_title("TEST - ufka gore hata"); ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)

onem = model.get_score(importance_type="gain")
top = sorted(onem.items(), key=lambda x: -x[1])[:15][::-1]
ax[1].barh([k for k, _ in top], [v for _, v in top])
ax[1].set_title("XGBoost - ozellik onemi (gain)"); ax[1].tick_params(labelsize=7)

ax[2].scatter([nn_mg[m] for m in ortak], [x_test["magaza_smape"][m] for m in ortak], s=4, alpha=.4)
lim = [0, max(max(nn_mg.values()), max(x_test["magaza_smape"].values()))]
ax[2].plot(lim, lim, "k--", lw=1)
ax[2].set_xlabel("NN sMAPE %"); ax[2].set_ylabel("XGBoost sMAPE %")
ax[2].set_title(f"Magaza bazli hata (r={korelasyon:.2f})"); ax[2].grid(alpha=.3)

plt.tight_layout(); plt.savefig(f"{CIKTI}/xgb_kiyas.png", dpi=130)
print(f"\ngrafik -> {CIKTI}/xgb_kiyas.png")



grafik -> /content/drive/MyDrive/Colab Notebooks/datasets/rossman/model/xgb_kiyas.png


In [10]:
# ------------------------------------------------------------------
# 8. KAYIT
# ------------------------------------------------------------------
with open(f"{CIKTI}/xgb_kiyas.json", "w") as f:
    json.dump({"val": x_val, "test": x_test,
               "en_iyi_round": int(model.best_iteration + 1),
               "egitim_dk": float(sure), "parametreler": par,
               "ozellikler": OZELLIK, "kantil_egitildi": bool(kantil),
               "ozellik_onemi": {k: float(v) for k, v in onem.items()},
               "magaza_hata_korelasyonu": korelasyon,
               "not": "XGBoost 88 origin, NN 264 origin; takvim araligi ayni"}, f, indent=2)
print(f"kaydedildi -> {CIKTI}/xgb_kiyas.json")

kaydedildi -> /content/drive/MyDrive/Colab Notebooks/datasets/rossman/model/xgb_kiyas.json
